In [1]:
import tkinter as tk
from tkinter import ttk, messagebox
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

# ------------------------
# Main Window
# ------------------------
root = tk.Tk()
root.title("Stock Price Prediction System")
root.geometry("750x650")
root.configure(bg="#F4F7FC")
root.resizable(False, False)

# ------------------------
# Header
# ------------------------
title = tk.Label(
    root,
    text="STOCK PRICE PREDICTION SYSTEM",
    font=("Arial", 22, "bold"),
    bg="#1E3A8A",
    fg="white",
    pady=15
)
title.pack(fill="x")

# ------------------------
# Company Selection
# ------------------------
frame = tk.Frame(root, bg="#F4F7FC")
frame.pack(pady=20)

tk.Label(frame, text="Company",
         font=("Arial",12,"bold"),
         bg="#F4F7FC").grid(row=0,column=0,padx=10,pady=10)

company = ttk.Combobox(frame,width=20,
                       values=["AAPL","GOOGL","AMZN","TSLA"])
company.grid(row=0,column=1)
company.current(0)

# ------------------------
# Open Price
# ------------------------
tk.Label(frame,text="Open Price",
         font=("Arial",12,"bold"),
         bg="#F4F7FC").grid(row=1,column=0,pady=10)

open_entry=tk.Entry(frame,width=25)
open_entry.grid(row=1,column=1)

# ------------------------
# High Price
# ------------------------
tk.Label(frame,text="High Price",
         font=("Arial",12,"bold"),
         bg="#F4F7FC").grid(row=2,column=0,pady=10)

high_entry=tk.Entry(frame,width=25)
high_entry.grid(row=2,column=1)

# ------------------------
# Low Price
# ------------------------
tk.Label(frame,text="Low Price",
         font=("Arial",12,"bold"),
         bg="#F4F7FC").grid(row=3,column=0,pady=10)

low_entry=tk.Entry(frame,width=25)
low_entry.grid(row=3,column=1)

# ------------------------
# Volume
# ------------------------
tk.Label(frame,text="Volume",
         font=("Arial",12,"bold"),
         bg="#F4F7FC").grid(row=4,column=0,pady=10)

volume_entry=tk.Entry(frame,width=25)
volume_entry.grid(row=4,column=1)

# ------------------------
# Prediction Label
# ------------------------
prediction=tk.Label(root,
                    text="Predicted Closing Price",
                    font=("Arial",16,"bold"),
                    bg="#F4F7FC",
                    fg="blue")

prediction.pack(pady=20)

# ------------------------
# Predict Function
# ------------------------
def predict():

    try:

        stock=company.get()

        model=joblib.load(f"models/{stock}_model.pkl")

        data=[[

            float(open_entry.get()),
            float(high_entry.get()),
            float(low_entry.get()),
            float(volume_entry.get())

        ]]

        price=model.predict(data)

        prediction.config(
            text=f"Predicted Closing Price : ${price[0]:.2f}"
        )

    except Exception as e:

        messagebox.showerror("Error",str(e))

# ------------------------
# Graph Function
# ------------------------
def show_graph():

    try:

        stock = company.get()

        data = pd.read_csv(f"datasets/{stock}.csv")

        model = joblib.load(f"models/{stock}_model.pkl")

        X = data[['Open','High','Low','Volume']]
        y = data['Close']

        predicted = model.predict(X)

        # New Window
        graph_window = tk.Toplevel(root)
        graph_window.title(f"{stock} Stock Graph")
        graph_window.geometry("900x600")
        graph_window.configure(bg="white")

        title = tk.Label(
            graph_window,
            text=f"{stock} Actual vs Predicted Closing Price",
            font=("Arial",18,"bold"),
            bg="white"
        )
        title.pack(pady=10)

        fig = plt.Figure(figsize=(10,5), dpi=100)
        ax = fig.add_subplot(111)

        ax.plot(y.values,
                label="Actual",
                color="blue",
                linewidth=2)

        ax.plot(predicted,
                label="Predicted",
                color="red",
                linewidth=2)

        ax.set_xlabel("Days")
        ax.set_ylabel("Closing Price")
        ax.set_title("Actual vs Predicted")
        ax.legend()
        ax.grid(True)

        canvas = FigureCanvasTkAgg(fig, master=graph_window)
        canvas.draw()
        canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

        close_btn = tk.Button(
            graph_window,
            text="Close",
            bg="red",
            fg="white",
            font=("Arial",12,"bold"),
            command=graph_window.destroy
        )
        close_btn.pack(pady=10)

    except Exception as e:
        messagebox.showerror("Error", str(e))
        
# ------------------------
# Clear Function
# ------------------------
def clear():

    open_entry.delete(0,tk.END)
    high_entry.delete(0,tk.END)
    low_entry.delete(0,tk.END)
    volume_entry.delete(0,tk.END)

    prediction.config(
        text="Predicted Closing Price"
    )

# ------------------------
# Buttons
# ------------------------
btn_frame=tk.Frame(root,bg="#F4F7FC")
btn_frame.pack(pady=20)

predict_btn=tk.Button(
    btn_frame,
    text="Predict",
    bg="#2563EB",
    fg="white",
    font=("Arial",12,"bold"),
    width=12,
    command=predict
)

predict_btn.grid(row=0,column=0,padx=10)

graph_btn=tk.Button(
    btn_frame,
    text="Show Graph",
    bg="green",
    fg="white",
    font=("Arial",12,"bold"),
    width=12,
    command=show_graph
)

graph_btn.grid(row=0,column=1,padx=10)

clear_btn=tk.Button(
    btn_frame,
    text="Clear",
    bg="orange",
    fg="white",
    font=("Arial",12,"bold"),
    width=12,
    command=clear
)

clear_btn.grid(row=0,column=2,padx=10)

exit_btn=tk.Button(
    btn_frame,
    text="Exit",
    bg="red",
    fg="white",
    font=("Arial",12,"bold"),
    width=12,
    command=root.destroy
)

exit_btn.grid(row=0,column=3,padx=10)

# ------------------------
# Status
# ------------------------
status=tk.Label(
    root,
    text="Ready",
    bd=1,
    relief=tk.SUNKEN,
    anchor="w",
    bg="white"
)

status.pack(side=tk.BOTTOM,fill=tk.X)

root.mainloop()